# RAG-Enhanced Coaching

**Purpose:** Extends the AI financial coaching system so it can ground its recommendations in a curated collection of financial-education content, not just the user's own numbers.

**Context, for anyone starting here.** This notebook is one piece of a larger AI financial-coaching system; the profile analytics behind it (persona clustering, behavioral scoring, spending forecasts, benchmarking) are produced upstream and simply consumed here as a `profile` dict. What this notebook adds is the retrieval layer: a small knowledge base of financial-education articles, embedded into a vector store, that the coach searches before answering a question. Combining that retrieved context with the user's own financial profile lets the coach give advice that is both grounded in source material and specific to the person asking.

**Design Notes:**

1. Two knowledge domains are seeded to start: emergency funds and debt reduction
  - Both are common early-stage coaching topics, and both have a natural progression (starter fund up through one, three, and six months of expenses; smallest-balance-first versus highest-interest-first repayment) that benefits from being grounded in a fixed reference rather than left to the model's general knowledge
2. Coverage is intentionally partial
  - Questions outside the knowledge base fall back to the model's general financial knowledge rather than being blocked, so the coach still answers, just without a source-grounded citation
3. The knowledge base and the coaching logic are built as two separable stages (§1, §2), so new domains can be added to the vector store without touching the coaching pipeline

In [1]:
# Load Libraries & Configure Client
import os
import re

import pandas as pd
from IPython.display import display

from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# OPENAI_API_KEY is read from a local .env file (not committed); LangChain's
# OpenAIEmbeddings and ChatOpenAI pick it up from the environment directly.
load_dotenv()


---
## 1 · Knowledge Base Processing & Vector Store

**Purpose:** Turns the knowledge base's source files into searchable embeddings, so the coaching pipeline in §2 can retrieve grounding context instead of relying on the model's unverified general knowledge.

**Design Notes:**

1. One unified Chroma collection (`financial_coach`) rather than one per topic
  - Retrieval stays cross-domain by construction: as more topics are added, a question can pull relevant context from any of them without the pipeline needing to know which collection to search
2. Each source file carries its own identity in its header: a domain, a set of tags, and `##`-delimited subsections
  - Splitting on those subsections turns each file into topic-sized chunks rather than embedding the whole document at once, so retrieval returns the specific concept a question is about, not the entire article
  - Every chunk is tagged with its source filename, so a document can be replaced (`replace_knowledge_document`) by deleting only the chunks with that source and re-adding the updated file, without rebuilding the collection
3. Coverage today is two files; the pipeline itself is domain-agnostic and expects new files to follow the same header/section convention


In [2]:
# Create Vector Store
embeddings = OpenAIEmbeddings()

vectorstore = Chroma(
    collection_name='financial_coach',
    embedding_function=embeddings,
    persist_directory='chroma_db',
)


In [3]:
# Knowledge Base Processing Pipeline
def load_knowledge_document(file_path):
    # Load Document
    loader = TextLoader(file_path)
    text = loader.load()[0].page_content

    # Domain
    domain_match = re.search(r'Domain:\s*(.+)', text)
    domain = domain_match.group(1).strip() if domain_match else 'Unknown'

    # Tags
    tags_match = re.search(r'Tags:\s*(.*?)\n\nContent Type:', text, re.DOTALL)
    tags = []
    if tags_match:
        tags = [
            tag.strip()
            for tag in tags_match.group(1).replace('\n', ' ').split(',')
            if tag.strip()
        ]

    # Source File
    source = os.path.basename(file_path)

    # Split Into Sections
    sections = re.split(r'\n##\s+', text)[1:]

    # Build Chunks
    chunks = []
    for section in sections:
        lines = section.strip().split('\n')
        topic = lines[0].strip()

        content = '\n'.join(lines[1:]).strip()
        content = re.sub(r'\n?-{3,}\s*$', '', content).strip()  # drop trailing section rule

        chunks.append(Document(
            page_content=content,
            metadata={
                'domain': domain,
                'topic': topic,
                'source': source,
                'tags': tags,
            },
        ))

    print(f'{source}: {len(chunks)} chunks created.')
    return chunks


# Add New Document
def add_knowledge_document(file_path, vectorstore):
    chunks = load_knowledge_document(file_path)
    vectorstore.add_documents(chunks)

    filename = os.path.basename(file_path)
    print(f'{filename}: {len(chunks)} chunks added.')
    print(f'Total Chunks: {vectorstore._collection.count()}\n')


# Replace Existing Document
def replace_knowledge_document(file_path, vectorstore):
    source = os.path.basename(file_path)

    # Remove Existing Chunks, Reload Updated Document
    vectorstore._collection.delete(where={'source': source})
    add_knowledge_document(file_path, vectorstore)
    print(f'{source} replaced.\n')


In [4]:
# Add Documents
add_knowledge_document('knowledge_base/01_emergency_fund.txt', vectorstore)
add_knowledge_document('knowledge_base/02_debt_reduction.txt', vectorstore)


01_emergency_fund.txt: 10 chunks created.
01_emergency_fund.txt: 10 chunks added.
Total Chunks: 10

02_debt_reduction.txt: 14 chunks created.
02_debt_reduction.txt: 14 chunks added.
Total Chunks: 24



In [5]:
# Validate Vector Store
all_docs = vectorstore.get()
metadata_df = pd.DataFrame(all_docs['metadatas'])

display(metadata_df['source'].value_counts())


source
02_debt_reduction.txt    14
01_emergency_fund.txt    10
Name: count, dtype: int64

**Observations:**

1. `01_emergency_fund.txt` splits into 10 chunks and `02_debt_reduction.txt` into 14, one per `##` subsection in each source file
2. Chunk counts track each file's outline directly: adding or removing a `##` subsection changes the chunk count on the next load, nothing else needs to change


In [6]:
# Inspect a Chunk
all_docs = vectorstore.get()

for doc, meta in zip(all_docs['documents'], all_docs['metadatas']):
    if meta['source'] == '01_emergency_fund.txt':
        print('Metadata:')
        print(meta)

        print('\nContent:')
        print(doc)
        break


---
## 2 · RAG-Enhanced Financial Coaching

**Purpose:** Answers a user's personal-finance question by combining retrieved knowledge-base context (when the topic is covered) with their own financial profile, so the response is both source-grounded and personalized.

**Design Notes:**

1. Every question is classified first, into `emergency_fund`, `debt_reduction`, `other_finance`, or `not_finance`
  - The two knowledge-base topics retrieve grounding context via vector search before generation; `other_finance` falls back to the model's general knowledge instead of being blocked; `not_finance` is declined outright, keeping the coach inside personal-finance topic boundaries
2. The user's financial profile (income, spending, debt, emergency fund status, and forecast) is folded into every response regardless of which path was used, so advice is never generic even when it isn't knowledge-base-grounded
3. Grounding responses in curated content for the two covered topics reduces hallucination risk on exactly the questions where a wrong answer, an invented emergency fund target, for example, would be most consequential


### 2.1 Test Profile

**Purpose:** Stands in for the upstream profile analytics (see context above) with one representative user: a "Near Successful" persona carrying a thin surplus, no emergency fund, and modest debt.


In [7]:
# Test Profile: Near Successful Persona
profile = {
    'persona': {
        'name': 'Near Successful',
        'cluster_id': 1,
        'description': (
            'Users whose spending patterns closely resemble financially successful '
            'users but who generate insufficient monthly surplus.'
        ),
    },

    'behavioral_alignment': {
        'success_similarity': 0.8259,
        'score': 82.6,
    },

    'financial_capacity': {
        'monthly_income': 4811.00,
        'avg_monthly_spend': 4731.66,
        'monthly_surplus': 79.34,
        'surplus_ratio': 0.0165,
        'score': 1.6,
    },

    'financial_stability': {
        'total_debt': 2262.00,
        'debt_to_income': 0.0392,
    },

    'emergency_fund': {
        'balance': 0.00,
        'target': 14433.00,
        'progress': 0.0669,
    },

    'forecasting': {
        'predicted_next_month_spending': 4942.56,
        'forecast_surplus': -131.56,
    },

    'benchmark': {
        'monthly_savings_opportunity': 381.00,
    },

    'spending_categories': [
        {'category': 'Shopping', 'amount': 650.00},
        {'category': 'Transportation', 'amount': 525.00},
        {'category': 'Food Discretionary', 'amount': 480.00},
    ],
}


### 2.2 Coaching Pipeline

**Purpose:** Classifies the question, retrieves knowledge-base context when the topic is covered, and generates a personalized response grounded in whichever source, knowledge base or general knowledge, applies.

**Design Notes:**

1. Classification and generation both run on `gpt-4o-mini` at a moderate temperature (0.5), favoring consistent, on-topic responses over creative variation
2. The generation prompt states explicitly that knowledge-base context, when present, takes priority over the model's own knowledge and must not be contradicted
3. Responses are capped at 250 words to keep coaching output actionable rather than exhaustive


In [8]:
# Coaching Pipeline: Classify, Retrieve, Generate
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.5)


# Topic Classification
def classify_question(question):
    prompt = ChatPromptTemplate.from_template('''
        You are a financial topic classifier.

        Classify the user's question into ONE category:
        - emergency_fund
        - debt_reduction
        - other_finance
        - not_finance

        Return ONLY the category name.

        User Question: {question}
    ''')

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({'question': question}).strip().lower()


# Retrieve Knowledge Base Context
def retrieve_context(question, vectorstore, k=4):
    results = vectorstore.similarity_search(question, k=k)
    context = '\n\n'.join(doc.page_content for doc in results)
    sources = list({doc.metadata.get('source', 'Unknown') for doc in results})
    return context, sources


# Profile Summary
def profile_summary(profile):
    return f'''
        Persona: {profile['persona']['name']}

        Monthly Income: ${profile['financial_capacity']['monthly_income']:,.0f}
        Monthly Spending: ${profile['financial_capacity']['avg_monthly_spend']:,.0f}
        Monthly Surplus: ${profile['financial_capacity']['monthly_surplus']:,.0f}

        Emergency Fund Balance: ${profile['emergency_fund']['balance']:,.0f}
        Emergency Fund Target: ${profile['emergency_fund']['target']:,.0f}

        Total Debt: ${profile['financial_stability']['total_debt']:,.0f}
        Debt-to-Income Ratio: {profile['financial_stability']['debt_to_income']:.1%}

        Forecast Surplus: ${profile['forecasting']['forecast_surplus']:,.0f}

        Potential Monthly Savings Opportunity:
        ${profile['benchmark']['monthly_savings_opportunity']:,.0f}
    '''


# Financial Coach
def ask_financial_coach(profile, question, vectorstore):
    category = classify_question(question)

    # Non-Finance Questions
    if category == 'not_finance':
        return 'I can only assist with personal finance related questions.'

    # Knowledge Base Topics
    if category in ('emergency_fund', 'debt_reduction'):
        context, sources = retrieve_context(question, vectorstore)
        source_type = 'Knowledge Base'

    # Other Finance Topics
    else:
        context, sources, source_type = '', [], 'General Financial Knowledge'

    prompt = ChatPromptTemplate.from_template('''
        You are a professional financial coach.

        User Financial Profile: {profile}

        Information Source: {source_type}

        Knowledge Base Context: {context}

        User Question: {question}

        Requirements:
        - Answer only personal finance questions.
        - If Knowledge Base Context is provided,
          rely primarily on that information.
        - Do not invent facts that contradict
          the knowledge base.
        - If no knowledge base context is provided,
          answer using general financial knowledge.
        - Personalize advice using the user's profile.
        - Focus on practical, actionable guidance.
        - Maintain a professional and supportive tone.
        - Limit responses to 250 words or less.
    ''')

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({
        'profile': profile_summary(profile),
        'source_type': source_type,
        'context': context,
        'question': question,
    })


In [9]:
# Quick Test
response = ask_financial_coach(
    profile=profile,
    question='What is the best way to start an emergency fund?',
    vectorstore=vectorstore,
)

print(response)


The best way to start an emergency fund is to set a specific, manageable savings goal and create a consistent plan to reach it. Given your current financial profile, here are some actionable steps:

1. **Set a Target**: Aim for a starter emergency fund of $500 to $1,000. This initial amount can cover many common emergencies and serves as a vital first milestone.

2. **Identify Savings Opportunities**: You have a potential monthly savings opportunity of $381. Consider allocating a portion of this toward your emergency fund. Even setting aside $100–$200 monthly can help you reach your goal quickly.

3. **Open a Dedicated Account**: Choose a high-yield savings account or a money market account to keep your emergency fund separate from your regular savings. This ensures easy access while earning some interest.

4. **Automate Savings**: If possible, set up an automatic transfer to your emergency fund account each month. This makes saving easier and lessens the temptation to spend that money

### 2.3 Example Behaviors

**Purpose:** Walks through the four classification paths, knowledge-base retrieval (two topics), general financial knowledge, and topic-boundary enforcement, against the same test profile.


In [10]:
# Example: Emergency Fund (Knowledge Base)
response = ask_financial_coach(
    profile, 'How much should I keep in my emergency fund?', vectorstore
)

print(response)


Given your financial profile, your current situation shows that you have no emergency fund, and your monthly spending is close to your income. It's crucial to establish a financial safety net to protect against unexpected expenses.

For your emergency fund, I recommend starting with a target of **$500 to $1,000**. This amount can cover many common emergencies, such as minor car repairs or medical expenses. Once you reach this starter emergency fund, you can aim to build it up to at least **one month of essential living expenses**, which would be around **$4,732** based on your current spending.

Since you have a potential monthly savings opportunity of **$381**, you can prioritize building your emergency fund. If you set aside **$250** each month, you can reach your starter emergency fund goal in just two months. After that, you can continue saving to build toward one month of expenses within the next few months.

Having this emergency fund will help prevent you from relying on debt du

In [11]:
# Example: Debt Reduction (Knowledge Base)
response = ask_financial_coach(
    profile, 'Should I use debt snowball or debt avalanche?', vectorstore
)

print(response)


Given your financial profile, choosing between the debt snowball and debt avalanche methods depends on your personal priorities and motivations.

**Debt Snowball Method:** This strategy focuses on paying off the smallest debts first. This approach can provide quick wins, boosting your motivation as you eliminate debts one by one. Since you have a total debt of $2,262, if you have smaller balances, you might find it encouraging to pay them off quickly.

**Debt Avalanche Method:** This approach prioritizes debts with the highest interest rates, which can save you more money in the long run. If any of your debts carry high interest, this method might be more financially efficient.

Given your current financial situation, where your monthly surplus is only $79 and you have a potential savings opportunity of $381, consider a hybrid approach. Start with the smallest balance to build momentum but also keep an eye on any higher interest debts. This way, you can make progress while staying moti

In [12]:
# Example: Other Finance (General Knowledge)
response = ask_financial_coach(
    profile, 'How does a Roth IRA work?', vectorstore
)

print(response)


A Roth IRA (Individual Retirement Account) is a retirement savings account that allows you to contribute after-tax income, meaning you pay taxes on the money before you deposit it into the account. The key benefits of a Roth IRA include:

1. **Tax-Free Growth**: Your investments grow tax-free, and you won’t pay taxes on withdrawals in retirement, provided you meet certain conditions.

2. **Flexible Withdrawals**: You can withdraw your contributions (but not earnings) at any time without penalties, making it a flexible option for saving.

3. **No Required Minimum Distributions (RMDs)**: Unlike traditional IRAs, Roth IRAs do not require you to take distributions at a certain age, allowing your savings to grow longer.

For your financial situation, consider contributing to a Roth IRA once you establish your emergency fund and reduce your debt. With a monthly surplus of $79 and a potential savings opportunity of $381, you could prioritize building your emergency fund first to cover unexpec

In [13]:
# Example: Not Finance (Topic Boundary)
response = ask_financial_coach(
    profile, 'How do I make lasagna?', vectorstore
)

print(response)


I can only assist with personal finance related questions.


**Observations:**

1. The two knowledge-base topics return specific, source-grounded guidance, a $500-$1,000 starter emergency fund target and a snowball-versus-avalanche comparison, drawn from the retrieved chunks rather than the model's own knowledge
2. The `other_finance` question (Roth IRA) is still answered and still personalized to the profile, just without knowledge-base grounding or sources
3. The `not_finance` question is declined outright, holding the topic boundary regardless of profile context
